In [152]:
import openseespywin as ops
import opsvis as opsv 
import numpy as np
import ipywidgets as widgets
import os
import matplotlib.pyplot as plt
import math
import opstool as opst
import eurocodepy as ecpy
import time as tt
# import openseespy.postprocessing.Get_Rendering as opsplt

In [153]:
import openseespy.opensees as ops

#Soil Element Properties
elasto_mat_tag = 1
thickness = 1.0
type = "PlaneStrain"

# Soil Material properties
E = 200e6         # Elastic modulus in Pa
nu = 0.3          # Poisson's ratio
rho = 0.0         # Density

# Storage for created nodes and elements
created_nodes = []
created_elements = []
block_registry = {}

def generate_block(start_x, start_y, length, height, node_offset, elem_offset, mat_tag, box_width, box_height, block_name="block"):
    num_x = int(length / box_width)
    num_y = int(height / box_height)

    def block_node_id(i, j):
        return node_offset + j * (num_x + 1) + i + 1

    block_nodes = []
    block_elements = []

    # Create nodes
    for j in range(num_y + 1):
        for i in range(num_x + 1):
            nid = block_node_id(i, j)
            x = start_x + i * box_width
            y = start_y + j * box_height
            ops.node(nid, x, y)
            created_nodes.append((nid, x, y))
            block_nodes.append(nid)

    # Create elements
    eid = elem_offset
    for j in range(num_y):
        for i in range(num_x):
            n1 = block_node_id(i, j)
            n2 = block_node_id(i + 1, j)
            n3 = block_node_id(i + 1, j + 1)
            n4 = block_node_id(i, j + 1)

            ops.element("quad", eid, n1, n2, n3, n4, thickness, type, mat_tag)
            created_elements.append((eid, n1, n2, n3, n4))
            block_elements.append(eid)
            eid += 1

    # Register block
    block_registry[block_name] = {
        "nodes": block_nodes,
        "elements": block_elements,
    }

    return (node_offset + (num_x + 1) * (num_y + 1), elem_offset + num_x * num_y)

# Start model
ops.wipe()
ops.model("Basic", "-ndm", 2, "-ndf", 2)

# Define material
ops.nDMaterial("ElasticIsotropic", elasto_mat_tag, E, nu, rho)

# Define contact material
contact_mat = 2
ops.nDMaterial("ContactMaterial2D", contact_mat, 0.1, 1000.0, 0.0, 0.0)

# Generate one block starting from node 1 and element 1
node_offset = 0
elem_offset = 1
node_offset, elem_offset = generate_block(45, 16, 8.0, 2.5, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="OAE_Soil_block")
#node_offset, elem_offset = generate_block(8.5, 0.25, 3.0, 7.5, node_offset, elem_offset, elasto_mat_tag, 0.5 , 0.5, block_name="F1_Soil_block")


In [154]:
opst.vis.plotly.plot_model()

In [155]:
# Fixity definitions
fixXY = [1, 1]
fixXonly = [1, 0]
fixYonly = [0, 1]

# Inputs
start_id = 1
end_id = 17
step = 1

# Generate node list
bottom_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in bottom_nodes:
    ops.fix(nid, *fixXY)


In [156]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

103


In [157]:
opst.vis.plotly.plot_model()

In [158]:
# Inputs
start_id = 18
end_id = 86
step = 17

# Generate node list
fix_left_nodes = list(range(start_id, end_id + 1, step))

# Apply fixity
for nid in fix_left_nodes:
    ops.fix(nid, *fixYonly)

In [159]:
opst.vis.plotly.plot_model()

In [160]:
ops.model("Basic", "-ndm", 2, "-ndf", 3)

In [161]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

103


In [162]:
sheet_pile_node = [
[53.000 , 31.300],
[52.500 , 31.280],
[52.004 , 31.220],
[51.514 , 31.122],
[51.033 , 30.985],
[50.565 , 30.810],
[50.112 , 30.599],
[49.677 , 30.352],
[49.263 , 30.072],
[48.873 , 29.760],
[48.508 , 29.417],
[48.172 , 29.047],
[47.866 , 28.652],
[47.593 , 28.233],
[47.354 , 27.795],
[47.150 , 27.338],
[46.983 , 26.867],
[46.854 , 26.384],
[46.763 , 25.893],
[46.712 , 25.395],
[46.700 , 24.896],
[46.728 , 24.396],
[46.796 , 23.901],
[46.903 , 23.413],
[47.048 , 22.934],
[47.230 , 22.469],
[47.449 , 22.020],
[47.703 , 21.589],
[47.990 , 21.180],
[48.308 , 20.795],
[48.656 , 20.436],
[49.032 , 20.106],
[49.432 , 19.807],
[49.855 , 19.540],
[50.298 , 19.308],
[50.758 , 19.112],
[51.232 , 18.953],
[51.717 , 18.832],
[52.210 , 18.749],
[52.708 , 18.706],
[53.000 , 18.700],
]

start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(sheet_pile_node):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["sheet_pile_nodes"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [163]:
opst.vis.plotly.plot_model()

In [164]:
ops.fix(143, 0, 1, 0)

In [165]:
opst.vis.plotly.plot_model()

In [166]:
master_node_ids = block_registry["sheet_pile_nodes"]["nodes"]
print(master_node_ids)

[103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143]


In [167]:
# Inputs
lagrange_start_id = 200
num_lagrange_nodes = 40
x_coord = 53.0
y_coord = 18.7

# Output list
lagrange_node_ids = []

# Create nodes
for i in range(num_lagrange_nodes):
    nid = lagrange_start_id + i
    ops.node(nid, x_coord, y_coord)
    created_nodes.append((nid, x_coord, y_coord))
    lagrange_node_ids.append(nid)

# Register block
block_registry["lagrange_nodes"] = {
    "nodes": lagrange_node_ids,
    "elements": []
}

In [168]:
lagrange_node_ids = block_registry["lagrange_nodes"]["nodes"]
print(lagrange_node_ids)

[200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239]


In [169]:
existing_tags = ops.getNodeTags()
start_id = max(existing_tags) + 1 if existing_tags else 1
print(start_id)

240


In [170]:
slave_node = [
[52.500 , 31.500],
[52.000 , 31.500],
[51.673 , 31.414],
[51.179 , 31.292],
[50.713 , 31.137],
[50.203 , 30.923],
[49.762 , 30.694],
[49.321 , 30.419],
[48.910 , 30.116],
[48.537 , 29.794],
[48.172 , 29.426],
[47.850 , 29.047],
[47.542 , 28.622],
[47.261 , 28.157],
[47.034 , 27.704],
[46.833 , 27.207],
[46.674 , 26.698],
[46.500 , 26.000],
[46.500 , 25.500],
[46.500 , 25.000],
[46.500 , 24.500],
[46.500 , 24.000],
[46.611 , 23.552],
[46.741 , 23.068],
[46.923 , 22.555],
[47.126 , 22.102],
[47.379 , 21.636],
[47.664 , 21.200],
[47.959 , 20.817],
[48.314 , 20.423],
[48.700 , 20.058],
[49.109 , 19.730],
[49.547 , 19.433],
[49.959 , 19.198],
[50.398 , 18.988],
[50.917 , 18.790],
[51.391 , 18.650],
]


start_id = start_id
segment_node_ids = []

for i, (x, y) in enumerate(slave_node):
    nid = start_id + i
    ops.node(nid, x, y)
    created_nodes.append((nid, x, y))
    segment_node_ids.append(nid)

# Register segment_lining as a separate block
block_registry["slave_nodes"] = {
    "nodes": segment_node_ids,
    "elements": []
}

In [171]:
slaveNode_A = block_registry["slave_nodes"]["nodes"]
print(master_node_ids)

[103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143]


In [172]:
final_slave_nodes = slaveNode_A + [100, 101,102]

In [173]:
master_nodes = master_node_ids
slave_nodes = final_slave_nodes
lagrange_nodes = lagrange_node_ids

contact_elem_ids = []
beam_start_id = 1000
for i in range(len(slave_nodes)):
    tag = beam_start_id + i
    iN = master_nodes[i]
    jN = master_nodes[i + 1]
    sN = slave_nodes[i]
    IN = lagrange_nodes[i]

    ops.element("BeamContact2D", tag, iN, jN, sN, IN, contact_mat, 0.5, 1e-10, 1e-10)
    created_elements.append((tag, iN, jN, sN, IN))
    contact_elem_ids.append(tag)

block_registry["contact_beams_right"] = {
    "nodes": master_nodes + slave_nodes + lagrange_nodes,
    "elements": contact_elem_ids
}

In [174]:
opst.vis.plotly.plot_model()

In [175]:
# Inputs
beam_nodes = master_node_ids  # Example node list (ordered start → end)
transFTag = 1
beam_secTag = 1
intTag = 401
Nint = 3
beam_start_id = 3000  # Starting element tag


# Geometry transformation and section definition
ops.geomTransf("Linear", transFTag)
ops.section("Elastic", beam_secTag, 200e6, 0.5, 0.000975)
ops.beamIntegration("Legendre", intTag, beam_secTag, Nint)

# Create elements
beam_elem_ids = []
for i in range(len(beam_nodes) - 1):
    sN = beam_nodes[i]
    eN = beam_nodes[i + 1]
    eid = beam_start_id + i
    ops.element("dispBeamColumn", eid, sN, eN, transFTag, intTag)
    created_elements.append((eid, sN, eN))
    beam_elem_ids.append(eid)

# Register block
block_registry["beam_elements"] = {
    "nodes": beam_nodes,
    "elements": beam_elem_ids
}

In [176]:
opst.vis.plotly.plot_model()